In [1]:
import pandas as pd
import numpy as np
import os

path = r"D:\Swapnil\Work\Projects\FIFA\Updated Base Files"

df = pd.read_csv(os.path.join(path, "Step8_Final_Dataset.csv"))

print(df.shape)

(1389312, 55)


In [2]:
def normalize(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    
    if series.max() == series.min():
        return series * 0
    
    return (series - series.min()) / (series.max() - series.min())

In [3]:
df["revenue_score"] = normalize(df["fifa_total_sales"])
df["profit_score"] = normalize(df["fifa_operating_profit"])
df["roas_score"] = normalize(df["roas"])
df["conversion_score"] = normalize(df["conversion_rate"])
df["customer_value_score"] = normalize(df["purchase_amount"])
df["loyalty_score"] = normalize(df["previous_purchases"])

In [4]:
df["opportunity_score"] = (
    0.30 * df["revenue_score"] +
    0.20 * df["profit_score"] +
    0.20 * df["roas_score"] +
    0.10 * df["conversion_score"] +
    0.10 * df["customer_value_score"] +
    0.10 * df["loyalty_score"]
)

In [5]:
threshold = df["opportunity_score"].quantile(0.75)

df["high_opportunity"] = (df["opportunity_score"] >= threshold).astype(int)

print(df["high_opportunity"].value_counts())

high_opportunity
0    1041984
1     347328
Name: count, dtype: int64


In [6]:
summary = df.groupby(
    ["company_name", "country", "region", "tier"],
    as_index=False
).agg({
    "fifa_total_sales": "sum",
    "fifa_operating_profit": "sum",
    "roas": "mean",
    "conversion_rate": "mean",
    "opportunity_score": "mean"
})

summary = summary.sort_values("opportunity_score", ascending=False)

summary["rank"] = range(1, len(summary) + 1)

In [7]:
top10 = summary.head(10)

print(top10)

    company_name      country         region    tier  fifa_total_sales  \
74         Pepsi       Mexico  North America    High      1.926422e+07   
26     Coca-Cola       Mexico  North America    High      2.027812e+07   
55         Pepsi       Canada  North America    High      1.712375e+07   
114     Red Bull      Germany         Europe    High      1.460025e+07   
102     Red Bull       Brazil  South America    High      1.518426e+07   
4      Coca-Cola      Belgium         Europe    High      1.427580e+07   
113     Red Bull       France         Europe    High      1.460025e+07   
77         Pepsi  New Zealand        Oceania  Medium      1.084504e+07   
64         Pepsi      England         Europe    High      1.541137e+07   
97      Red Bull    Argentina  South America    High      1.351875e+07   

     fifa_operating_profit      roas  conversion_rate  opportunity_score  rank  
74            6.517046e+06  0.262310         0.179362           0.283153     1  
26            7.083745e

In [8]:
df.to_csv(os.path.join(path, "Step9_Final_Dataset.csv"), index=False)

summary.to_excel(os.path.join(path, "Step9_Summary.xlsx"), index=False)

In [9]:
print(df.columns)

Index(['sales_id', 'date', 'retailer', 'retailer_id', 'us_region', 'state',
       'city', 'product', 'price_per_unit', 'units_sold', 'total_sales',
       'operating_profit', 'operating_margin', 'country_id', 'country',
       'region', 'confederation', 'is_world_cup_team', 'is_host', 'tier',
       'country_multiplier', 'company_name', 'sim_units_sold',
       'sim_price_per_unit', 'sim_total_sales', 'sim_operating_profit',
       'match_day_flag', 'match_count', 'fifa_total_sales', 'fifa_units_sold',
       'fifa_operating_profit', 'knockout_stage_flag', 'fifa_sales_uplift_pct',
       'customer_id', 'age', 'gender', 'category', 'purchase_amount',
       'location', 'season', 'review_rating', 'discount_applied',
       'promo_code_used', 'previous_purchases', 'payment_method',
       'purchase_frequency', 'age_group', 'campaign_spend', 'impressions',
       'clicks', 'conversions', 'discount_percent', 'ctr', 'conversion_rate',
       'roas', 'revenue_score', 'profit_score', 'roas_sc

In [10]:
print(df["opportunity_score"].describe())

count    1.389312e+06
mean     1.976998e-01
std      7.207834e-02
min      3.445820e-03
25%      1.482789e-01
50%      1.907821e-01
75%      2.376401e-01
max      8.828305e-01
Name: opportunity_score, dtype: float64


In [11]:
print(df["high_opportunity"].value_counts())

high_opportunity
0    1041984
1     347328
Name: count, dtype: int64


In [12]:
print(summary.head())

    company_name  country         region  tier  fifa_total_sales  \
74         Pepsi   Mexico  North America  High      1.926422e+07   
26     Coca-Cola   Mexico  North America  High      2.027812e+07   
55         Pepsi   Canada  North America  High      1.712375e+07   
114     Red Bull  Germany         Europe  High      1.460025e+07   
102     Red Bull   Brazil  South America  High      1.518426e+07   

     fifa_operating_profit      roas  conversion_rate  opportunity_score  rank  
74            6.517046e+06  0.262310         0.179362           0.283153     1  
26            7.083745e+06  0.358668         0.132241           0.262282     2  
55            6.256364e+06  0.061420         0.176130           0.259040     3  
114           6.788589e+06  0.037624         0.173801           0.251443     4  
102           7.060133e+06  0.041244         0.167857           0.249359     5  
